In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from pathlib import Path

# ============================================================
# CONFIG
# ============================================================

RANDOM_STATE = 42
OUTPUT_DIR = Path("D:\\Deep Learning\\preprocessed_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

PADUFES_FILE = "D:\\Deep Learning\\metadata.csv"

ISIC_TRAIN_GT_FILE = "D:\\Deep Learning\\ISIC_2019_Training_GroundTruth.csv"
ISIC_TRAIN_META_FILE = "D:\\Deep Learning\\ISIC_2019_Training_Metadata.csv"

ISIC_TEST_GT_FILE = "D:\\Deep Learning\\ISIC_2019_Test_GroundTruth.csv"
ISIC_TEST_META_FILE = "D:\\Deep Learning\\ISIC_2019_Test_Metadata.csv"

MCR_UNIFIED_FILE = "D:\\Deep Learning\\MCR-SL_dataset\\unified_diagnosis.xlsx"
MCR_LESION_FILE = "D:\\Deep Learning\\MCR-SL_dataset\\lesion.xlsx"
MCR_SUBJECT_FILE = "D:\\Deep Learning\\MCR-SL_dataset\\subject.xlsx"
MCR_IMAGE_FILE = "D:\\Deep Learning\\MCR-SL_dataset\\image.xlsx"
MCR_DERM_FILE = "D:\\Deep Learning\\MCR-SL_dataset\\dermatology_diagnosis.xlsx"
MCR_HISTO_FILE = "D:\\Deep Learning\\MCR-SL_dataset\\histopathology_diagnosis.xlsx"

# Core known classes learned from PAD-UFES
KNOWN_CLASSES = ["AK", "BCC", "MEL", "NEV", "SCC", "SK"]

# Unknown classes for target-domain open-world evaluation
ISIC_UNKNOWN_CLASSES = ["ANG", "DF", "UNK"]
MCR_UNKNOWN_CLASSES = ["ANG", "ATY", "DF"]

INCLUDE_MCR_UNK = False
if INCLUDE_MCR_UNK:
    MCR_UNKNOWN_CLASSES = MCR_UNKNOWN_CLASSES + ["UNK"]


# ============================================================
# LABEL HARMONIZATION
# ============================================================

LABEL_MAP = {
    "MEL": "MEL",
    "BCC": "BCC",
    "SCC": "SCC",
    "DF": "DF",
    "ATY": "ATY",
    "UNK": "UNK",

    "NV": "NEV",
    "NEV": "NEV",
    "NEVUS": "NEV",

    "AK": "AK",
    "ACK": "AK",
    "ACTINIC_KERATOSIS": "AK",

    "SK": "SK",
    "SEK": "SK",
    "BKL": "SK",
    "SEBORRHEIC_KERATOSIS": "SK",
    "BENIGN_KERATOSIS": "SK",

    "ANG": "ANG",
    "VASC": "ANG",
    "ANGIOMA": "ANG",

    "BOWEN_CARCINOMA": "SCC",
}

def normalize_label(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip().upper()
    x = x.replace("-", "_").replace("/", "_").replace(" ", "_")
    return x

def harmonize_label(x):
    x = normalize_label(x)
    if pd.isna(x):
        return np.nan
    return LABEL_MAP.get(x, x)


# ============================================================
# UTILITY FUNCTIONS
# ============================================================

def safe_text(x):
    if pd.isna(x):
        return np.nan
    return str(x).strip()

def onehot_to_label(df, label_cols):
    def row_to_label(row):
        active = [c for c in label_cols if row[c] == 1 or row[c] == 1.0]
        if len(active) == 1:
            return active[0]
        elif len(active) == 0:
            return np.nan
        else:
            return "|".join(active)
    return df[label_cols].apply(row_to_label, axis=1)

def split_source(df, label_col="label_harmonized"):
    train_df, val_df = train_test_split(
        df,
        test_size=0.20,
        random_state=RANDOM_STATE,
        stratify=df[label_col]
    )

    train_df = train_df.copy()
    val_df = val_df.copy()

    train_df["split"] = "source_train"
    val_df["split"] = "source_val"

    return train_df, val_df

def split_target_train_pool_for_adapt_val(
    df,
    known_classes,
    unknown_classes,
    label_col="label_harmonized"
):
    """
    Used for ISIC official TRAINING file.

    Logic:
    - target_adapt gets known classes only.
    - target_val gets known + unknown classes.
    - No target_test is created from the training file.
    """

    keep_classes = known_classes + unknown_classes
    df = df[df[label_col].isin(keep_classes)].copy()

    df["known_unknown"] = np.where(
        df[label_col].isin(known_classes),
        "known",
        "unknown"
    )

    known_df = df[df["known_unknown"] == "known"].copy()
    unknown_df = df[df["known_unknown"] == "unknown"].copy()

    # Split known target data into adapt and validation.
    known_adapt, known_val = train_test_split(
        known_df,
        test_size=0.20,
        random_state=RANDOM_STATE,
        stratify=known_df[label_col]
    )

    # Unknown target training-pool samples are used only in validation,
    # never in adaptation.
    target_adapt = known_adapt.copy()
    target_val = pd.concat([known_val, unknown_df], ignore_index=True)

    target_adapt = target_adapt.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
    target_val = target_val.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

    target_adapt["split"] = "target_adapt"
    target_val["split"] = "target_val"

    return target_adapt, target_val

def prepare_official_target_test(
    df,
    known_classes,
    unknown_classes,
    label_col="label_harmonized"
):
    """
    Used for official ISIC test file.
    Keeps known + unknown classes and marks known/unknown.
    """

    keep_classes = known_classes + unknown_classes
    df = df[df[label_col].isin(keep_classes)].copy()

    df["known_unknown"] = np.where(
        df[label_col].isin(known_classes),
        "known",
        "unknown"
    )

    df["split"] = "target_test"
    df = df.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

    return df

def split_target_no_official_test(
    df,
    known_classes,
    unknown_classes,
    label_col="label_harmonized"
):
    """
    Used for MCR-SL because it has no official train/test partition.

    Logic:
    - known target samples: 60/20/20 into adapt/val/test
    - unknown target samples: no adaptation, split into val/test only
    """

    keep_classes = known_classes + unknown_classes
    df = df[df[label_col].isin(keep_classes)].copy()

    df["known_unknown"] = np.where(
        df[label_col].isin(known_classes),
        "known",
        "unknown"
    )

    target_known = df[df["known_unknown"] == "known"].copy()
    target_unknown = df[df["known_unknown"] == "unknown"].copy()

    # Known classes: 60/20/20
    known_adapt, known_temp = train_test_split(
        target_known,
        test_size=0.40,
        random_state=RANDOM_STATE,
        stratify=target_known[label_col]
    )

    known_val, known_test = train_test_split(
        known_temp,
        test_size=0.50,
        random_state=RANDOM_STATE,
        stratify=known_temp[label_col]
    )

    # Unknown classes: no adaptation, split into val/test only.
    unknown_val_parts = []
    unknown_test_parts = []

    for cls, grp in target_unknown.groupby(label_col):
        grp = grp.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
        n = len(grp)

        if n == 1:
            val_part = grp.iloc[:0]
            test_part = grp.iloc[:1]
        else:
            n_val = n // 2
            val_part = grp.iloc[:n_val]
            test_part = grp.iloc[n_val:]

        unknown_val_parts.append(val_part)
        unknown_test_parts.append(test_part)

    unknown_val = (
        pd.concat(unknown_val_parts, ignore_index=True)
        if unknown_val_parts else pd.DataFrame(columns=df.columns)
    )

    unknown_test = (
        pd.concat(unknown_test_parts, ignore_index=True)
        if unknown_test_parts else pd.DataFrame(columns=df.columns)
    )

    target_adapt = known_adapt.copy()
    target_val = pd.concat([known_val, unknown_val], ignore_index=True)
    target_test = pd.concat([known_test, unknown_test], ignore_index=True)

    target_adapt = target_adapt.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
    target_val = target_val.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
    target_test = target_test.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

    target_adapt["split"] = "target_adapt"
    target_val["split"] = "target_val"
    target_test["split"] = "target_test"

    return target_adapt, target_val, target_test

def report(df, name):
    print(f"\n{'='*80}")
    print(name)
    print(f"{'='*80}")
    print("Shape:", df.shape)

    if "label_harmonized" in df.columns:
        print("\nClass distribution:")
        print(df["label_harmonized"].value_counts().sort_index())

    if "known_unknown" in df.columns:
        print("\nKnown/Unknown:")
        print(df["known_unknown"].value_counts())

def keep_cols(df, cols):
    return df[[c for c in cols if c in df.columns]].copy()


# ============================================================
# 1. LOAD PAD-UFES SOURCE
# ============================================================

pad = pd.read_csv(PADUFES_FILE)

pad["dataset"] = "PAD-UFES-20"
pad["role"] = "source"
pad["sample_id"] = pad["img_id"]
pad["image_id"] = pad["img_id"]
pad["image_file"] = pad["img_id"]

pad["label_raw"] = pad["diagnostic"]
pad["label_harmonized"] = pad["label_raw"].apply(harmonize_label)

pad["age_clean"] = pad["age"]
pad["sex_clean"] = pad["gender"].apply(safe_text)
pad["site_clean"] = pad["region"].apply(safe_text)

pad["diameter_1_clean"] = pad["diameter_1"]
pad["diameter_2_clean"] = pad["diameter_2"]

pad["known_unknown"] = "known"

pad_source = pad[pad["label_harmonized"].isin(KNOWN_CLASSES)].copy()

source_train, source_val = split_source(pad_source)


# ============================================================
# 2. LOAD ISIC OFFICIAL TRAINING TARGET POOL
# ============================================================

isic_train_gt = pd.read_csv(ISIC_TRAIN_GT_FILE)
isic_train_meta = pd.read_csv(ISIC_TRAIN_META_FILE)

isic_label_cols = ["MEL", "NV", "BCC", "AK", "BKL", "DF", "VASC", "SCC", "UNK"]

isic_train_gt["label_raw"] = onehot_to_label(isic_train_gt, isic_label_cols)
isic_train_gt["label_harmonized"] = isic_train_gt["label_raw"].apply(harmonize_label)

isic_train = isic_train_gt.merge(isic_train_meta, on="image", how="left")

isic_train["dataset"] = "ISIC 2019"
isic_train["role"] = "target"
isic_train["sample_id"] = isic_train["image"]
isic_train["image_id"] = isic_train["image"]
isic_train["image_file"] = isic_train["image"].astype(str) + ".jpg"

isic_train["age_clean"] = isic_train["age_approx"]
isic_train["sex_clean"] = isic_train["sex"].apply(safe_text)
isic_train["site_clean"] = isic_train["anatom_site_general"].apply(safe_text)

isic_adapt, isic_val = split_target_train_pool_for_adapt_val(
    isic_train,
    known_classes=KNOWN_CLASSES,
    unknown_classes=ISIC_UNKNOWN_CLASSES
)


# ============================================================
# 3. LOAD ISIC OFFICIAL TEST TARGET SET
# ============================================================

isic_test_gt = pd.read_csv(ISIC_TEST_GT_FILE)
isic_test_meta = pd.read_csv(ISIC_TEST_META_FILE)

# Some ISIC test files include score_weight and validation_weight.
# They are preserved if present.
isic_test_gt["label_raw"] = onehot_to_label(isic_test_gt, isic_label_cols)
isic_test_gt["label_harmonized"] = isic_test_gt["label_raw"].apply(harmonize_label)

isic_test = isic_test_gt.merge(isic_test_meta, on="image", how="left")

isic_test["dataset"] = "ISIC 2019"
isic_test["role"] = "target"
isic_test["sample_id"] = isic_test["image"]
isic_test["image_id"] = isic_test["image"]
isic_test["image_file"] = isic_test["image"].astype(str) + ".jpg"

isic_test["age_clean"] = isic_test["age_approx"]
isic_test["sex_clean"] = isic_test["sex"].apply(safe_text)
isic_test["site_clean"] = isic_test["anatom_site_general"].apply(safe_text)

isic_test = prepare_official_target_test(
    isic_test,
    known_classes=KNOWN_CLASSES,
    unknown_classes=ISIC_UNKNOWN_CLASSES
)


# ============================================================
# 4. LOAD MCR-SL TARGET
# ============================================================

mcr_unified = pd.read_excel(MCR_UNIFIED_FILE)
mcr_lesion = pd.read_excel(MCR_LESION_FILE)
mcr_subject = pd.read_excel(MCR_SUBJECT_FILE)
mcr_image = pd.read_excel(MCR_IMAGE_FILE)
mcr_derm = pd.read_excel(MCR_DERM_FILE)
mcr_histo = pd.read_excel(MCR_HISTO_FILE)

mcr = mcr_unified.copy()

mcr["label_raw"] = mcr["unified_diagnosis"]
mcr["label_harmonized"] = mcr["label_raw"].apply(harmonize_label)

mcr = mcr.merge(
    mcr_lesion,
    on="lesion_id",
    how="left",
    suffixes=("", "_lesion")
)

mcr = mcr.merge(
    mcr_subject,
    on="subject_id",
    how="left",
    suffixes=("", "_subject")
)

mcr["dataset"] = "MCR-SL"
mcr["role"] = "target"
mcr["sample_id"] = mcr["lesion_id"]

# Current default: diagnosis image id.
# Later you can modify this to use clinical images or multiple images per lesion.
mcr["image_id"] = mcr["diagnosis_image_id"]
mcr["image_file"] = mcr["diagnosis_image_id"]

mcr["age_clean"] = mcr["age"]
mcr["sex_clean"] = mcr["sex"].apply(safe_text)
mcr["site_clean"] = mcr["location_group"].apply(safe_text)
mcr["diameter_clean"] = mcr["diameter"]

mcr_adapt, mcr_val, mcr_test = split_target_no_official_test(
    mcr,
    known_classes=KNOWN_CLASSES,
    unknown_classes=MCR_UNKNOWN_CLASSES
)


# ============================================================
# 5. SELECT FINAL COLUMNS
# ============================================================

COMMON_COLS = [
    "dataset", "role", "split",
    "sample_id", "image_id", "image_file",
    "label_raw", "label_harmonized", "known_unknown",
    "age_clean", "sex_clean", "site_clean"
]

PAD_EXTRA_COLS = [
    "patient_id", "lesion_id",
    "smoke", "drink", "pesticide",
    "skin_cancer_history", "cancer_history",
    "fitspatrick",
    "diameter_1_clean", "diameter_2_clean",
    "itch", "grew", "hurt", "changed", "bleed", "elevation",
    "biopsed"
]

ISIC_EXTRA_COLS = [
    "lesion_id", "age_approx", "anatom_site_general", "sex",
    "score_weight", "validation_weight"
]

MCR_EXTRA_COLS = [
    "lesion_id", "subject_id",
    "referral_diagnosis",
    "lesion_status_when_captured",
    "location", "location_group",
    "diameter_clean",
    "malignancy",
    "natural_hair_color",
    "skin_reaction_to_sun",
    "moles_body_18", "moles_bigger_5mm", "moles_bigger_20cm", "moles_body",
    "sunburn_number", "sunburn_number_group",
    "sunbed",
    "h_cancer", "h_skin_cancer", "h_skin_cancer_relatives",
    "organ_transplant", "immunosuppresion"
]

source_train_final = keep_cols(source_train, COMMON_COLS + PAD_EXTRA_COLS)
source_val_final = keep_cols(source_val, COMMON_COLS + PAD_EXTRA_COLS)

isic_adapt_final = keep_cols(isic_adapt, COMMON_COLS + ISIC_EXTRA_COLS)
isic_val_final = keep_cols(isic_val, COMMON_COLS + ISIC_EXTRA_COLS)
isic_test_final = keep_cols(isic_test, COMMON_COLS + ISIC_EXTRA_COLS)

mcr_adapt_final = keep_cols(mcr_adapt, COMMON_COLS + MCR_EXTRA_COLS)
mcr_val_final = keep_cols(mcr_val, COMMON_COLS + MCR_EXTRA_COLS)
mcr_test_final = keep_cols(mcr_test, COMMON_COLS + MCR_EXTRA_COLS)


# ============================================================
# 6. REPORT RESULTS
# ============================================================

report(source_train_final, "SOURCE TRAIN: PAD-UFES")
report(source_val_final, "SOURCE VAL: PAD-UFES")

report(isic_adapt_final, "TARGET ADAPT: ISIC OFFICIAL TRAINING FILE")
report(isic_val_final, "TARGET VAL: ISIC OFFICIAL TRAINING FILE")
report(isic_test_final, "TARGET TEST: ISIC OFFICIAL TEST FILE")

report(mcr_adapt_final, "TARGET ADAPT: MCR-SL")
report(mcr_val_final, "TARGET VAL: MCR-SL")
report(mcr_test_final, "TARGET TEST: MCR-SL")


# ============================================================
# 7. SAVE OUTPUTS
# ============================================================

source_train_final.to_csv(OUTPUT_DIR / "padufes_source_train.csv", index=False)
source_val_final.to_csv(OUTPUT_DIR / "padufes_source_val.csv", index=False)

isic_adapt_final.to_csv(OUTPUT_DIR / "isic_target_adapt.csv", index=False)
isic_val_final.to_csv(OUTPUT_DIR / "isic_target_val.csv", index=False)
isic_test_final.to_csv(OUTPUT_DIR / "isic_target_test_official.csv", index=False)

mcr_adapt_final.to_csv(OUTPUT_DIR / "mcr_target_adapt.csv", index=False)
mcr_val_final.to_csv(OUTPUT_DIR / "mcr_target_val.csv", index=False)
mcr_test_final.to_csv(OUTPUT_DIR / "mcr_target_test.csv", index=False)

combined = pd.concat(
    [
        source_train_final,
        source_val_final,
        isic_adapt_final,
        isic_val_final,
        isic_test_final,
        mcr_adapt_final,
        mcr_val_final,
        mcr_test_final,
    ],
    ignore_index=True
)

combined.to_csv(OUTPUT_DIR / "all_preprocessed_splits.csv", index=False)

print("\nPreprocessing complete.")
print(f"Saved files to: {OUTPUT_DIR.resolve()}")